<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/ParthSharma_AI_Agents_Exercise_Set_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents — Exercise Set 2

**Topic:** ReAct Agents, Custom Tools, Agent Debugging, Memory (LangChain + Groq)

**Instructions:**
* All problems here are NEW (different from Set 1).
* Part A = trace & predict (no coding), Part B = fix broken code, Part C = build, Part D = mini-project.
* Run the setup cells first.

---

## ⚙️ Setup (run first)

In [1]:
!pip install -q langchain langchain-groq langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("GROQ API KEY: ")

GROQ API KEY: ··········


In [3]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import math

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
print("LLM ready:", llm.model_name)

LLM ready: llama-3.3-70b-versatile


---
# Part A — Trace & Predict 🔍 (No coding, brain only)

For each question, **write down what you think the agent will do BEFORE running anything.**

### A1. Predict the Trace

The agent has tools: `calculator`, `get_weather`, `knowledge_base` (from class).

User asks: **"What is a neural network, and what is 2 to the power 10?"**

Predict the full trace:
* How many Action steps? Which tools, in which order?
* What will each Action Input be?

**Your prediction:**

```
Thought: User asking two things:
1.What is neural network?(knowledge question)
2.What is 2^10?(calculation)
Action: knowledge_base
Action Input: "neural network"
Observation: A neural network is a machine learning model inspired by the human brain,consisting of interconnected nodes that learns pattern from the data.

Thought: Now calculate 2^10
Action: calculator
Action Input: "2^10"
Observation: 1024
Thought: I have answers to both questions
...
Final Answer: neural network"
Observation: A neural network is a machine learning model inspired by the human brain,consisting of interconnected nodes that learns pattern from the data. 2^10=1024.
```

### A2. Spot the Ambiguity

User asks: **"Tell me about React."**

The class `knowledge_base` has an entry for `"react"` (the ReAct prompting strategy). But the user might mean React.js (the JavaScript library)!

1. Which answer will our agent give, and why?
2. How would you rewrite the tool's docstring OR the kb entry to reduce this confusion?

**Your answer:**

...

1. The agent will return the ReAct prompting strategy becuase the knowledge_base contains an entry for "react". It matches the query literally and has no way to know that the user might instead mean React.js, unless the tool description or knowledge_base is more specific.

2.Make the entry more specific instad of using the ambiguous term "react".Use like "react_prompting" and a description like "Information about ReAct(reason +act)"


OR

improve tool's docstring like "this knowledge base contains llm concepts such as ReAct propting, chainOfThoughts".


### A3. The Format Rules

Look at the ReAct prompt from class. It says: *"Action must be exactly one of the tool names listed."*

1. What happens if the LLM writes `Action: Calculator` (capital C) when the tool is named `calculator`?
2. Which `AgentExecutor` parameter helps the agent recover from this kind of mistake?

**Your answer:**

...

1. The agent will fail to find a tool named Calculator because tool names are matched exactly. Since only calculator exists, the action will be treated as invail tool name, which will result in an error or failure.

2. the "handle_parsing_roors=True" parameter allows the agent to recover form passing or formatting errors such in this case by feeding the error back to the llm and giving it another chance to produce a correctly formatted action.

---
# Part B — Fix the Broken Code 🔧

Each cell below has bugs. **Find them, fix them, and write a one-line comment `# BUG:` explaining each fix.**

### B1. Broken Tool (2 bugs)

Hint: one bug prevents the agent from ever choosing this tool; the other crashes the lookup.

In [6]:
# FIX THIS CELL — 2 bugs

@tool
def pincode_lookup(city: str) -> str:

  '''
  Input is city name and return corressponding pin code
  '''

  #BUG: Delhi won't be executed because all cities are in lower characters so changing the case of input city
  city = city.lower()
  pin_db = {
        "delhi": "110001",
        "mumbai": "400001",
        "sonipat": "131001",
        "bengaluru": "560001",
  }
  #BUG:If city not found it will give key error instead of not found
  return pin_db.get(city,"Pincode not found")

# Test after fixing:
print(pincode_lookup.invoke("Delhi"))       # should work (capital D!)
print(pincode_lookup.invoke("Chennai"))     # should NOT crash

110001
Pincode not found


### B2. Broken Agent Setup (3 bugs)

Hint: check the prompt variables, the executor arguments, and what's missing to stop runaway loops.

In [9]:
# FIX THIS CELL — 3 bugs

broken_prompt = PromptTemplate.from_template("""You are a helpful AI assistant.

You have access to these tools:
{tools}
Use the following format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
...
Thought: I now know the final answer
Final Answer: the final answer

Question: {input}
Thought:{agent_scratchpad}
""")

broken_agent = create_react_agent(llm=llm, tools=[pincode_lookup], prompt=broken_prompt)

broken_executor = AgentExecutor(
    agent=broken_agent,
    tools=[pincode_lookup],
    verbose=True,
    max_iterations=3,
    handle_parsing_errors=True,
)

# Test after fixing:
r = broken_executor.invoke({"input": "What is the pincode of Sonipat?"})



> Entering new AgentExecutor chain...
Thought: To find the pincode of Sonipat, I need to use the pincode_lookup function, which takes the city name as input and returns the corresponding pincode.

Action: pincode_lookup
Action Input: Sonipat131001Thought: I now know the final answer
Final Answer: The pincode of Sonipat is 131001.

> Finished chain.


**Write your 5 bug explanations here:**

B1 bug 1: ...
B1 bug 2: ...
B2 bug 1: ...
B2 bug 2: ...
B2 bug 3: ...

Bug 1: City lookup was case-sensitive ("Delhi" ≠ "delhi"). Convert input to lowercase.

Bug 2: Unknown city caused a KeyError. Use dict.get() with a default message.




---


Bug 1: Prompt is missing required variables (tool_names and agent_scratchpad) for a ReAct agent.

Bug 2: AgentExecutor is missing the tools argument.

Bug 3: AgentExecutor is missing max_iterations, so the agent has no protection against runaway loops.

---
# Part C — Build New Tools 🛠️

### C1. Grade Calculator Tool

Build a tool `grade_calculator` that takes marks (out of 100, as a string) and returns the grade:

| Marks | Grade |
|-------|-------|
| 90+   | A     |
| 75–89 | B     |
| 60–74 | C     |
| 40–59 | D     |
| < 40  | Fail  |

Handle bad input (e.g. `"abc"` or `"150"`) gracefully with an error message.

In [11]:
@tool
def grade_calculator(marks: str) -> str:
    """
    Calculate the grade for marks out of 100.
    Input should be a number as a string between 0 and 100.
    Returns grades:
    A(>=90), B(89~75), C(74~60), D(59~40), Fail(<40).
    Returns ans error message for invalid input.
    """
    # TODO: convert to number, validate 0-100, return grade
    try:
      score=int(marks)
    except ValueError:
      return "Error: Marks must be a valid integer value"
    if score<0 or score>100:
      return "Error: Marks must be between 0 and 100"
    if score>=90:
        return "A"
    elif score>=75:
        return "B"
    elif score>=60:
        return "C"
    elif score>=40:
        return "D"
    else:
        return "Fail"

# Tests:
print(grade_calculator.invoke("92"))    # Grade: A
print(grade_calculator.invoke("55"))    # Grade: D
print(grade_calculator.invoke("150"))   # Error message
print(grade_calculator.invoke("abc"))   # Error message

A
D
Error: Marks must be between 0 and 100
Error: Marks must be a valid integer value


### C2. Unit Converter Tool (with parsing!)

Build `unit_converter` that handles input like `"5 km to miles"` or `"10 kg to pounds"`.

Support at least: km↔miles (1 km = 0.621 mi) and kg↔pounds (1 kg = 2.205 lb).

Hint: `.split()` the input string. This is harder than it looks — the agent sends free-form text!

In [12]:
@tool
def unit_converter(query: str) -> str:
    """
    # TODO: description — tell the agent the EXACT input format,
    # e.g. '<number> <unit> to <unit>' like '5 km to miles'
    Convert units.
    Input format must be:
    "<number> <unit> to <unit>"
    Examples:
    "5 km to miles"
    "10 kg to pounds"

    Supported conversions:
    - km to miles
    - kg to pounds
    """
    try:
      parts=query.lower().split()
      if len(parts)!=4 or parts[2]!="to":
        return "Error: Use format '<number> <unit> to <unit>'."
      value=float(parts[0])
      unit1=parts[1]
      unit2=parts[3]
      if unit1=="km" and unit2=="miles":
        return str(value*0.621)+" miles"
      elif unit1=="miles" and unit2=="km":
        return str(value*1.609)+" km"
      elif unit1=="kg" and unit2=="pounds":
        return str(value*2.205)+" pounds"
      elif unit1=="pounds" and unit2=="kg":
        return str(value*0.454)+" kg"
      else:
        return "Error: Unsupported conversion."
    except ValueError:
      return "Error: Invalid input format."
    except Exception as e:
      return f"Error: {e}"

# Tests:
print(unit_converter.invoke("5 km to miles"))     # ~3.11 miles
print(unit_converter.invoke("10 kg to pounds"))   # ~22.05 pounds

3.105 miles
22.05 pounds


### C3. Assemble + Stress Test

1. Build an agent with `grade_calculator`, `unit_converter`, and the class `calculator`.
2. Ask it this tricky question and observe:

> "I scored 78 out of 100 and my school is 3 km away. What is my grade and how far is that in miles?"

3. Then ask a question where the agent must **calculate first, then grade**:

> "I got 45, 60, and 75 in three tests (each out of 100). What is my average, and what grade is that?"

Does it chain the tools correctly? Note the order of Actions.

In [ ]:
# TODO: build agent + executor and run both questions

**Calculator is being built for the calculator agent.**

In [14]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Input examples:
    "2+3"
    "(45+60+75)/3"
    "2**10"
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

In [16]:
# ReAct Prompt
prompt = PromptTemplate.from_template("""
You are a helpful AI assistant.

You have access to the following tools:

{tools}

Use the following format:

Question: the question to answer
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the user

Question: {input}
Thought: {agent_scratchpad}
""")

# Create the ReAct Agent
agent = create_react_agent(
    llm=llm,
    tools=[calculator, grade_calculator, unit_converter],
    prompt=prompt
)

# Create the Executor
executor = AgentExecutor(
    agent=agent,
    tools=[calculator, grade_calculator, unit_converter],
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# **TestCase 1:**

In [17]:
response1 = executor.invoke(
    {
        "input": "I scored 78 out of 100 and my school is 3 km away. What is my grade and how far is that in miles?"
    }
)

print("\nFinal Response:\n")
print(response1["output"])



> Entering new AgentExecutor chain...
Thought: To solve this problem, I need to calculate the grade and convert the distance from kilometers to miles. First, I will calculate the grade using the grade_calculator function.

Action: grade_calculator
Action Input: 78BNow that I have the grade, I need to convert the distance from kilometers to miles using the unit_converter function.

Action: unit_converter
Action Input: 3 km to miles1.863 milesThought: I now know the final answer
Final Answer: Your grade is B and the distance to your school is 1.863 miles.

> Finished chain.

Final Response:

Your grade is B and the distance to your school is 1.863 miles.


**Thought:**
Need to determine the grade.

**Action:** grade_calculator
Action Input: 78

**Observation:**
Grade: B

**Thought:**
Now convert 3 km to miles.

**Action:** unit_converter

**Action Input:** 3 km to miles

**Observation:** 1.86 miles

**Final Answer:**
Your grade is B and 3 km is approximately 1.86 miles.

# **TestCase 2:**

In [18]:
response2 = executor.invoke(
    {
        "input": "I got 45, 60, and 75 in three tests (each out of 100). What is my average, and what grade is that?"
    }
)

print("\nFinal Response:\n")
print(response2["output"])



> Entering new AgentExecutor chain...
Thought: To find the average of the three test scores, I need to add them up and divide by the number of tests. Then, I can use the grade calculator to determine the grade based on the average score.

Action: calculator
Action Input: "(45+60+75)/3"60.0Thought: Now that I have the average score, I can use the grade calculator to determine the grade based on this average score.

Action: grade_calculator
Action Input: "60"CI now know the final answer
Final Answer: Your average score is 60.0, and your grade is C.

> Finished chain.

Final Response:

Your average score is 60.0, and your grade is C.


**Thought:**
First calculate the average.

**Action:** calculator

**Action Input:** (45+60+75)/3

**Observation:**
60

**Thought:**
Now calculate the grade.

**Action:** grade_calculator

**Action Input:** 60

**Observation:**

**Grade:** C

**Final Answer:**
Your average is 60 and your grade is C.

**Observations (order of tools used, any mistakes):**

----------------------

**Answer 1:**

grade_calculator

        ↓

unit_converter

        ↓

Final Answer

**Answer 2:**

calculator


        ↓

grade_calculator

        ↓

Final Answer

---
# Part D — Mini Project: Personal Study Assistant 🎓 (Graded)

Build a complete **Study Assistant Agent with memory** that has these 3 tools:

1. `syllabus_lookup(subject)` — dictionary of at least 4 subjects → topic lists (e.g. `"python": "variables, loops, functions, OOP"`).
2. `study_time(topic)` — dictionary of topics → estimated hours (e.g. `"loops": "3 hours"`).
3. `grade_calculator` — reuse from C1.

Then run this **4-turn conversation** with the SAME executor (memory must work):

1. `"Hi, I am <your name> and I want to learn Python."`
2. `"What topics are in the Python syllabus?"`
3. `"How many hours do I need for loops?"`
4. `"What was my name and which subject did I want to learn?"` ← memory test!

**Submit:** the full verbose output of turn 4.

In [20]:
# Mini project workspace

# TODO: Tool 1 — syllabus_lookup
@tool
def syllabus_lookup(subject: str) -> str:
    """
    Return the syllabus topics for a subject.
    Input should be a subject name.
    """

    syllabus = {
        "python": "Variables, Data Types, Operators, Loops, Functions, OOP",
        "java": "Variables, Loops, Arrays, OOP, Collections",
        "dbms": "ER Model, SQL, Normalization, Transactions",
        "dsa": "Arrays, Linked Lists, Trees, Graphs, Dynamic Programming"
    }

    return syllabus.get(subject.lower(), "Subject not found.")

# TODO: Tool 2 — study_time
@tool
def study_time(topic: str) -> str:
    """
    Return the estimated study time for a topic.
    """

    hours = {
        "variables": "2 hours",
        "data types": "2 hours",
        "operators": "1 hour",
        "loops": "3 hours",
        "functions": "4 hours",
        "oop": "6 hours",
        "arrays": "3 hours",
        "graphs": "8 hours",
        "trees": "7 hours",
        "sql": "5 hours"
    }

    return hours.get(topic.lower(), "Study time not available.")

# TODO: reuse grade_calculator
@tool
def grade_calculator(marks: str) -> str:
    """
    Calculate grade from marks (0-100).
    """

    try:
        marks = int(marks)
    except:
        return "Error: Marks must be an integer."

    if marks < 0 or marks > 100:
        return "Error: Marks must be between 0 and 100."

    if marks >= 90:
        return "Grade: A"
    elif marks >= 75:
        return "Grade: B"
    elif marks >= 60:
        return "Grade: C"
    elif marks >= 40:
        return "Grade: D"
    else:
        return "Grade: Fail"
# TODO: memory prompt (must include {chat_history})
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

prompt = PromptTemplate.from_template("""
You are a helpful study assistant.

Previous conversation:
{chat_history}

You have access to these tools:

{tools}

Use the following format:

Question: {input}
Thought: think about the problem.
Action: one of [{tool_names}]
Action Input: input for the tool
Observation: tool output
...(repeat if needed)
Thought: I now know the answer.
If you already know the answer from the conversation history and no tool is required, do NOT use an Action.
Simply write:

Thought: I already know the answer from the conversation history.
Final Answer: ...

Question: {input}
Thought:{agent_scratchpad}
""")
# TODO: agent + executor with memory
tools = [
    syllabus_lookup,
    study_time,
    grade_calculator
]

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)
# TODO: run the 4 turns
print("\n========== TURN 1 ==========\n")
executor.invoke({
    "input": "Hi, I am Rahul and I want to learn Python."
})
print("\n========== TURN 2 ==========\n")
executor.invoke({
    "input": "What topics are in the Python syllabus?"
})
print("\n========== TURN 3 ==========\n")
executor.invoke({
    "input": "How many hours do I need for loops?"
})
print("\n========== TURN 4 ==========\n")
result = executor.invoke({
    "input": "What was my name and which subject did I want to learn?"
})

print("\nFinal Output:")
print(result["output"])


========== TURN 1 ==========



> Entering new AgentExecutor chain...
Thought: To help Rahul learn Python, I first need to understand what topics are typically covered in a Python syllabus.

Action: syllabus_lookup
Action Input: PythonVariables, Data Types, Operators, Loops, Functions, OOPThought: Now that I have the syllabus topics for Python, I can see that it covers a range of fundamental concepts such as variables, data types, operators, loops, functions, and object-oriented programming (OOP). To help Rahul learn Python, I should estimate the study time required for each of these topics.

Action: study_time
Action Input: Variables2 hoursThought: I have estimated the study time for the "Variables" topic, which is 2 hours. Next, I should estimate the study time for the remaining topics in the Python syllabus, such as Data Types, Operators, Loops, Functions, and OOP.

Action: study_time
Action Input: Data Types2 hoursThought: I have estimated the study time for the "Data Types" topic

---
## ⭐ Bonus (Optional): One Tool to Rule Them All?

Instead of 3 separate tools, could you build ONE tool `study_helper(query)` that does everything with if/else inside?

Try it, then answer: **Why is this a BAD design for agents?** (Hint: think about what the docstring would have to say, and how the agent decides which tool to pick.)

**Your answer:**

...

The tool has to handle many unrelated tasks, making its docstring long and ambiguous.
The agent cannot easily determine which capability it should use.
It's harder to maintain and extend because every new feature goes into the same tool.
Separate tools (syllabus_lookup, study_time, grade_calculator) are more modular, reusable, and allow the agent to select the most appropriate tool for each task.

---
## ✅ Submission Checklist

- [ ] Part A: All 3 predictions/answers written
- [ ] Part B: All 5 bugs found, fixed, and explained
- [ ] Part C: Both tools pass their test cells; stress test observations written
- [ ] Part D: 4-turn memory conversation works, turn-4 output pasted
- [ ] Bonus (optional)

**Tip:** `Runtime → Restart and run all` before submitting.